Zapis do pliku odbywa się analogicznie do odczytu - większość formatów wektorowych zapiszemy za pomocą zunifikowanej metody to_file(). Struktura zapisywanych danych jest determinowana przez bibliotekę na podstawie zadeklarowanego rozszerzenia w nazwie pliku.

In [5]:
import geopandas as gpd

path_to_files = "dane/"
cities = gpd.read_file(path_to_files+"pl_miasta_wojewódzkie.geojson")

# Zapis do pliku, np jako .gpkg:
cities.to_file("dane.gpkg")

# Możemy też zamiast w formacie danych przestrzennych dane zapisać do CSV:
cities.to_csv("dane.csv", 
    index=False, 
    sep=";",
    decimal=",",
    encoding="utf-8")
cities.to_csv("dane.txt", index=False) # Mimo zmiany rozszerzenia na .txt używając metody to_csv() gwarantujemy strukturę tabelaryczną

# Czy odpalając dwa razy ten sam zapis plik się nadpisze, powstanie kopia czy program wyrzuci błąd?

In [6]:
# Przypomnienie - typ geometrii każdego rekordu w Gdf zbadać możemy przez:
print(cities.iloc[0].geometry.geom_type)

# Dla punktów sprawdzić możemy np:
print(cities.iloc[0].geometry)
print(cities.iloc[0].geometry.x) # Bezpośrednio x
print(cities.iloc[0].geometry.y) # Bezpośrednio y

# Możemy też te wartości przypisać do kolumny:
cities["x"] = cities.geometry.x
cities["y"] = cities.geometry.y
print(cities.head())

Point
POINT (19.45807945874724 51.76967922313351)
19.45807945874724
51.76967922313351
   gid     nazwa                                       źródło  \
0   10      Łódź  wektoryzacja własna na podstawie danych OSM   
1   11  Katowice  wektoryzacja własna na podstawie danych OSM   
2   12     Opole  wektoryzacja własna na podstawie danych OSM   
3   13   Wrocław  wektoryzacja własna na podstawie danych OSM   
4   14    Poznań  wektoryzacja własna na podstawie danych OSM   

               legenda liczba_ludności                   geometry          x  \
0  stolica województwa            None  POINT (19.45808 51.76968)  19.458079   
1  stolica województwa            None   POINT (19.03155 50.2591)  19.031546   
2  stolica województwa            None  POINT (17.92517 50.66639)  17.925171   
3  stolica województwa            None  POINT (17.03467 51.10752)  17.034674   
4  stolica województwa            None  POINT (16.93051 52.40633)  16.930507   

           y  
0  51.769679  
1  50.259105

Dla danych liniowych i poligonowych analogicznie możemy odczytywać ich statystyki:

In [7]:
rivers = gpd.read_file(path_to_files+"pl_rzeki.geojson")
zones = gpd.read_file(path_to_files+"pl_pasy_ukształtowania.geojson")

print(rivers.iloc[0].geometry.length) # W jakiej jednostce jest podana długość?
print(zones.iloc[0].geometry.length) # Co zwraca length dla poligonu?

print(zones.iloc[0].geometry.area) # Powierzchnia
print(zones.iloc[0].geometry.centroid) # Centroid

print(zones.iloc[0].geometry.bounds) # BB
print(zones.iloc[0].geometry.envelope) # Też BB - jaka jest różnica między bounds a envelope?

0.6146857787927802
12.793313778055868
1.8805435986439616
POINT (21.64839259964594 50.23465640797339)
(18.45398350911258, 49.74799567707629, 23.399675110546443, 50.82683180624041)
POLYGON ((18.45398350911258 49.74799567707629, 23.399675110546443 49.74799567707629, 23.399675110546443 50.82683180624041, 18.45398350911258 50.82683180624041, 18.45398350911258 49.74799567707629))


In [13]:
# Wiemy już że układ współrzędnych danych możemy sprawdzić za pomocą:
print(cities.crs)
# Możemy też:
print(cities.crs.name)
print(cities.crs.to_epsg())

# Jeśli dane nie mają przypisanego układu, można zrobić to poprzez:
cities.set_crs("EPSG:4326", inplace=True) # Co robi parametr inplace, dostępny przy wielu metodach gpd? 
print(cities.crs) 
# Dlaczego set_crs nie jest transformacją układu?

# Transformacja:
print(rivers.iloc[0].geometry.length)
rivers2 = rivers.to_crs("EPSG:2180")
print(rivers2.iloc[0].geometry.length)

EPSG:4326
WGS 84
4326
EPSG:4326
0.6146857787927802
51670.29094947464


In [11]:
# Niektóre struktury danych pozwalają na przechowywanie wielu warstw w jednym pliku. Na początek wymuśmy taki obiekt poprzez stworzenie pliku .gpkg
cities.to_file(
    "polska.gpkg",
    layer="miasta"
)

rivers.to_file(
    "polska.gpkg",
    layer="rzeki"
)

zones.to_file(
    "polska.gpkg",
    layer="pasy_ukształtowania"
)

# Możemy zbadać taki obiekt:
print(gpd.list_layers("polska.gpkg"))

# Jeśli jednak spróbujemy go załadować...
pl = gpd.read_file(
    "polska.gpkg"
) # Co będzie wynikiem takiej operacji?
print(pl.head())

# Możemy bezpośrednio wskazywać na warstwy które chcemy wczytywać:
rivers3 = gpd.read_file(
    "polska.gpkg",
    layer="rzeki"
)
print (rivers3.head())

                  name    geometry_type
0               miasta            Point
1                rzeki  MultiLineString
2  pasy_ukształtowania     MultiPolygon
   gid     nazwa                                       źródło  \
0   10      Łódź  wektoryzacja własna na podstawie danych OSM   
1   11  Katowice  wektoryzacja własna na podstawie danych OSM   
2   12     Opole  wektoryzacja własna na podstawie danych OSM   
3   13   Wrocław  wektoryzacja własna na podstawie danych OSM   
4   14    Poznań  wektoryzacja własna na podstawie danych OSM   

               legenda liczba_ludności          x          y  \
0  stolica województwa            None  19.458079  51.769679   
1  stolica województwa            None  19.031546  50.259105   
2  stolica województwa            None  17.925171  50.666385   
3  stolica województwa            None  17.034674  51.107525   
4  stolica województwa            None  16.930507  52.406332   

                    geometry  
0  POINT (19.45808 51.76968)  
1 

c:\Users\Krzysztof\anaconda3\Lib\site-packages\pyogrio\geopandas.py:382: UserWarning: More than one layer found in 'polska.gpkg': 'miasta' (default), 'rzeki', 'pasy_ukształtowania'. Specify layer parameter to avoid this warning.
  result = read_func(


In [12]:
from shapely import GeometryCollection, Point, LineString, Polygon
# Jaka jest relacja shapely do gdf?

geometry = GeometryCollection([
    Point(0, 0),
    LineString([(1, 1), (2, 2)]),
    Polygon([(3, 3), (3, 4), (4, 4), (4, 3)])
])

print(geometry)
print(geometry.geom_type)

# Możemy też zbudować w oparciu o taką geometrię instancję GeoDataFrame:
gdf = gpd.GeoDataFrame(
    geometry=[geometry],
    crs="EPSG:4326"
)
print(gdf)
print(gdf.geom_type)

# Lub:
gdf2 = gpd.GeoDataFrame(
    geometry=list(geometry.geoms),
    crs="EPSG:4326"
)
print(gdf2)
print(gdf2.geom_type)

GEOMETRYCOLLECTION (POINT (0 0), LINESTRING (1 1, 2 2), POLYGON ((3 3, 3 4, 4 4, 4 3, 3 3)))
GeometryCollection
                                            geometry
0  GEOMETRYCOLLECTION (POINT (0 0), LINESTRING (1...
0    GeometryCollection
dtype: object
                              geometry
0                          POINT (0 0)
1                LINESTRING (1 1, 2 2)
2  POLYGON ((3 3, 3 4, 4 4, 4 3, 3 3))
0         Point
1    LineString
2       Polygon
dtype: object
